# Ruta B: segmentación no supervisada con K-Means
**Plantilla para cualquier dataset** · Colab → Streamlit → GitHub → Docker → Google Cloud

Este notebook sigue la guía del reto, pero está preparado para trabajar con **cualquier CSV o Excel**. Las variables y la limpieza se eligen en una sola celda de configuración (o con un selector visual), y el notebook genera todos los archivos del proyecto.

**Cómo usarlo**
1. Ejecuta la sección 0 (preparación).
2. Carga tus datos (sección 1) y revisa el diagnóstico (sección 2).
3. Elige variables y limpieza (sección 3): edita `CONFIG` o usa el selector visual.
4. Elige k (4), entrena y evalúa (5), interpreta y nombra los clusters (6).
5. Guarda el modelo y genera `train.py`, `app.py`, Docker y README (7 a 11).
6. Prueba Streamlit (10), descarga el proyecto y súbelo a GitHub (12). Los comandos de GCP están al final (13).

Las celdas marcadas con ✏️ son las que normalmente tendrás que editar.

| Archivo generado | Para qué sirve |
|---|---|
| `cluster_utils.py` | Motor: carga, limpieza, pipeline, elección de k y perfiles (lo usan `train.py` y `app.py`) |
| `config.json` | Tus decisiones: variables, limpieza, k y nombres de los clusters |
| `train.py` | Reconstruye el modelo con `python train.py` |
| `model.joblib` | Pipeline entrenado + metadatos para la app |
| `app.py` | Interfaz Streamlit |
| `requirements.txt`, `Dockerfile`, `.dockerignore` | Empaquetado |
| `README.md` | Documentación (se llena sola; completa lo marcado con ✏️) |
| `data/datos.csv` | Copia del dataset |

## 0. Preparación (guía §4)

In [ ]:
%pip install -q streamlit scikit-learn pandas numpy joblib openpyxl matplotlib ipywidgets

In [ ]:
import sys, importlib
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# En Colab se trabaja en /content; en local, junto al notebook
BASE = Path("/content") if IN_COLAB else Path.cwd()
PROJECT = BASE / "ml_streamlit_project"
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print("Carpeta del proyecto:", PROJECT)
if IN_COLAB:
    print("⚠️ El almacenamiento de Colab es temporal: al final descarga el .zip o súbelo a GitHub.")

### Motor reutilizable: `cluster_utils.py`
Toda la lógica (carga, limpieza, pipeline, elección de k, perfiles) vive en este archivo y **no hace falta editarlo**.

¿Por qué en un archivo aparte y no en celdas? Porque el pipeline usa transformadores propios (por ejemplo, el que convierte `"$ 1.234.567"` en número). `joblib` guarda una *referencia* al módulo donde se definió cada clase. Si las clases estuvieran en el notebook, `model.joblib` no podría cargarse en `app.py` ni dentro de Docker.

In [ ]:
%%writefile {PROJECT}/cluster_utils.py
"""
cluster_utils.py - motor reutilizable de la Ruta B (clustering con K-Means).

Lo importan el notebook, train.py y app.py.

Las clases de limpieza viven aqui (y no en el notebook) a proposito: joblib
guarda una *referencia* al modulo donde se definio cada clase. Si estuvieran
definidas en el notebook, model.joblib no se podria cargar dentro de Docker.
"""
from __future__ import annotations

import re
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, RobustScaler, StandardScaler

# ---------------------------------------------------------------------------
# 1. Configuracion: todo lo que se puede elegir
# ---------------------------------------------------------------------------
DEFAULT_CONFIG = {
    # --- Variables ---------------------------------------------------------
    "numeric_features": "auto",     # "auto" o lista de columnas numericas
    "categorical_features": [],     # [] (ninguna), "auto" o lista de columnas
    "log_features": "auto",         # "auto", [] o lista: numericas muy sesgadas a las que se aplica log
    "exclude": [],                  # columnas que nunca deben entrar al modelo
    # --- Limpieza por filas (solo durante el entrenamiento) ---------------
    "drop_duplicates": True,        # elimina filas exactamente repetidas
    "max_missing_row": 0.5,         # elimina filas con mas de este % de faltantes en las variables elegidas
    # --- Criterios del modo "auto" -----------------------------------------
    "max_missing_col": 0.4,         # descarta columnas con mas de este % de faltantes
    "max_categories": 15,           # categoricas con mas niveles se descartan / se agrupan
    "skew_threshold": 1.0,          # asimetria a partir de la cual "auto" aplica log
    # --- Limpieza dentro del pipeline (se repite igual en la app) ---------
    "numeric_imputer": "median",    # median | mean | most_frequent
    "outliers": "iqr",              # iqr | quantile | none  (recorta, no elimina filas)
    "iqr_factor": 1.5,
    "quantiles": [0.01, 0.99],
    "scaler": "standard",           # standard | robust | minmax | none
    "categorical_imputer": "most_frequent",  # most_frequent | constant ("Desconocido")
    "normalize_categories": True,   # " bogotá " y "BOGOTÁ" -> "Bogotá"
    # --- Modelo ------------------------------------------------------------
    "k": "auto",                    # "auto" (mejor silhouette) o un entero
    "k_min": 2,
    "k_max": 8,
    "n_init": 10,
    "random_state": 42,
    "silhouette_sample": 5000,      # muestra para silhouette en datasets grandes
    "min_cluster_pct": 2.0,         # "auto" evita k con clusters de menos de este %
    # --- Interpretacion (si se dejan vacios se generan automaticamente) ---
    "title": "",
    "cluster_names": {},
    "cluster_descriptions": {},
}

_VALID_OPTIONS = {
    "numeric_imputer": {"median", "mean", "most_frequent"},
    "outliers": {"iqr", "quantile", "none"},
    "scaler": {"standard", "robust", "minmax", "none"},
    "categorical_imputer": {"most_frequent", "constant"},
}

NA_STRINGS = {
    "", "na", "n/a", "nan", "null", "none", "-", "--", "?", "nd", "n.d.",
    "s/d", "sin dato", "sin informacion", "sin información", "no aplica",
}


def merge_config(config: dict | None = None) -> dict:
    """Combina la configuracion del usuario con los valores por defecto y la valida."""
    cfg = deepcopy(DEFAULT_CONFIG)
    if config:
        unknown = set(config) - set(cfg)
        if unknown:
            raise ValueError(
                f"Opciones desconocidas en CONFIG: {sorted(unknown)}. "
                f"Opciones validas: {sorted(cfg)}"
            )
        cfg.update(deepcopy(config))
    for key, valid in _VALID_OPTIONS.items():
        if cfg[key] not in valid:
            raise ValueError(f"CONFIG['{key}'] = {cfg[key]!r} no es valido. Use uno de {sorted(valid)}.")
    if cfg["k"] != "auto":
        cfg["k"] = int(cfg["k"])
        if cfg["k"] < 2:
            raise ValueError("CONFIG['k'] debe ser 'auto' o un entero >= 2.")
    cfg["quantiles"] = [float(q) for q in cfg["quantiles"]]
    cfg["cluster_names"] = {int(k): str(v) for k, v in (cfg.get("cluster_names") or {}).items()}
    cfg["cluster_descriptions"] = {
        int(k): str(v) for k, v in (cfg.get("cluster_descriptions") or {}).items()
    }
    return cfg


# ---------------------------------------------------------------------------
# 2. Carga de datos (CSV con , o ; / Excel / URL / archivo subido)
# ---------------------------------------------------------------------------
def load_table(source, name: str | None = None, sheet_name=0) -> pd.DataFrame:
    """Lee CSV o Excel detectando separador y codificacion automaticamente."""
    label = (name or str(source)).lower().split("?")[0]
    if label.endswith((".xlsx", ".xls")):
        df = pd.read_excel(source, sheet_name=sheet_name)
    else:
        df = None
        for encoding in ("utf-8-sig", "latin-1"):
            try:
                if hasattr(source, "seek"):
                    source.seek(0)
                df = pd.read_csv(source, sep=None, engine="python", encoding=encoding)
                break
            except UnicodeDecodeError:
                continue
        if df is None:
            raise ValueError("No se pudo leer el archivo. Use CSV (UTF-8 o Latin-1) o Excel.")
    df.columns = [str(c).strip() for c in df.columns]
    return df


# ---------------------------------------------------------------------------
# 3. Conversion de valores "sucios"
# ---------------------------------------------------------------------------
_NOT_NUMBER = re.compile(r"[^\d,.\-eE+]")


def _parse_number(value) -> float:
    """'1.234,5' -> 1234.5 | '$ 2,000' -> 2000 | '15%' -> 15 | 'n/a' -> NaN."""
    if value is None:
        return np.nan
    if isinstance(value, (bool, np.bool_)):
        return float(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip()
    if text.lower() in NA_STRINGS:
        return np.nan
    negative = text.startswith("(") and text.endswith(")")
    text = _NOT_NUMBER.sub("", text)
    if "," in text and "." in text:
        if text.rfind(",") > text.rfind("."):          # 1.234,56 (formato colombiano)
            text = text.replace(".", "").replace(",", ".")
        else:                                           # 1,234.56
            text = text.replace(",", "")
    elif "," in text:
        parts = text.split(",")
        if len(parts) == 2 and len(parts[1]) != 3:      # 3,5 -> decimal
            text = text.replace(",", ".")
        else:                                           # 1,234,567 -> miles
            text = text.replace(",", "")
    elif text.count(".") > 1:                           # 1.234.567 -> miles
        text = text.replace(".", "")
    try:
        number = float(text)
    except ValueError:
        return np.nan
    return -number if negative else number


def to_number(values) -> pd.Series:
    """Convierte una columna a float; lo que no sea numero queda como NaN."""
    series = values if isinstance(values, pd.Series) else pd.Series(values)
    if pd.api.types.is_bool_dtype(series) or pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").astype(float)
    return series.map(_parse_number).astype(float)


def clean_category(value, normalize: bool = True):
    """Limpia una categoria: espacios, faltantes, '1.0' -> '1' y (opcional) mayusculas."""
    if value is None:
        return np.nan
    try:
        if pd.isna(value):
            return np.nan
    except (TypeError, ValueError):
        pass
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        value = int(value)
    text = " ".join(str(value).split())
    if text.lower() in NA_STRINGS:
        return np.nan
    return text.capitalize() if normalize else text


def _as_frame(X, columns=None) -> pd.DataFrame:
    if isinstance(X, pd.DataFrame):
        return X
    return pd.DataFrame(np.asarray(X, dtype=object), columns=columns)


# ---------------------------------------------------------------------------
# 4. Transformadores propios (van DENTRO del pipeline)
# ---------------------------------------------------------------------------
class NumericCleaner(BaseEstimator, TransformerMixin):
    """Convierte texto con formato de numero a float ('1.234,5', '$2.000', '15%')."""

    def fit(self, X, y=None):
        frame = _as_frame(X)
        self.columns_ = [str(c) for c in frame.columns]
        return self

    def transform(self, X):
        frame = _as_frame(X, self.columns_)
        if frame.shape[1] == 0:
            return np.empty((len(frame), 0))
        return np.column_stack([to_number(frame[c]).to_numpy(dtype=float) for c in frame.columns])


class CategoricalCleaner(BaseEstimator, TransformerMixin):
    """Convierte cada categoria a texto limpio y unifica los faltantes como NaN."""

    def __init__(self, normalize=True):
        self.normalize = normalize

    def fit(self, X, y=None):
        frame = _as_frame(X)
        self.columns_ = [str(c) for c in frame.columns]
        return self

    def transform(self, X):
        frame = _as_frame(X, self.columns_)
        out = np.empty(frame.shape, dtype=object)
        for j, col in enumerate(frame.columns):
            out[:, j] = [clean_category(v, self.normalize) for v in frame[col].tolist()]
        return out


class SignedLog1p(BaseEstimator, TransformerMixin):
    """log(1 + |x|) conservando el signo: reduce la asimetria de montos e ingresos."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        values = np.asarray(X, dtype=float)
        return np.sign(values) * np.log1p(np.abs(values))


class OutlierClipper(BaseEstimator, TransformerMixin):
    """Recorta valores extremos a limites aprendidos en el entrenamiento (no borra filas)."""

    def __init__(self, method="iqr", factor=1.5, quantiles=(0.01, 0.99)):
        self.method = method
        self.factor = factor
        self.quantiles = quantiles

    def fit(self, X, y=None):
        values = np.asarray(X, dtype=float)
        q_low, q_high = np.nanpercentile(values, [q * 100 for q in self.quantiles], axis=0)
        if self.method == "iqr":
            q1, q3 = np.nanpercentile(values, [25, 75], axis=0)
            iqr = q3 - q1
            lower, upper = q1 - self.factor * iqr, q3 + self.factor * iqr
            # Si IQR = 0 (columna casi constante) se usan los cuantiles para no aplanarla
            flat = iqr == 0
            lower = np.where(flat, q_low, lower)
            upper = np.where(flat, q_high, upper)
        else:
            lower, upper = q_low, q_high
        self.lower_ = np.asarray(lower, dtype=float)
        self.upper_ = np.asarray(upper, dtype=float)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)


# ---------------------------------------------------------------------------
# 5. Diagnostico de columnas y seleccion de variables
# ---------------------------------------------------------------------------
_ID_NAME = re.compile(
    r"(^id$|^id[_\W]|[_\W]id$|^idx|codigo|código|cedula|cédula|documento|^nit$|uuid|^index$|^unnamed)",
    re.IGNORECASE,
)


def _looks_like_date(series: pd.Series) -> bool:
    import warnings
    sample = series.dropna().astype(str).head(200)
    if sample.empty:
        return False
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            parsed = pd.to_datetime(sample, errors="coerce", format="mixed", dayfirst=True)
        except (TypeError, ValueError):
            return False
    return parsed.notna().mean() >= 0.9


def diagnose(df: pd.DataFrame, config: dict | None = None) -> pd.DataFrame:
    """Tabla con tipo detectado, % faltantes, valores unicos y una sugerencia por columna."""
    cfg = merge_config(config)
    rows = []
    n = len(df)
    for col in df.columns:
        series = df[col]
        non_null = int(series.notna().sum())
        missing = 1 - non_null / n if n else 1.0
        n_unique = int(series.nunique(dropna=True))
        skew = np.nan

        if pd.api.types.is_datetime64_any_dtype(series):
            kind = "fecha"
        elif pd.api.types.is_bool_dtype(series):
            kind = "binaria"
        elif pd.api.types.is_numeric_dtype(series):
            kind = "numérica"
        else:
            parsed = to_number(series)
            ok = parsed.notna().sum() / non_null if non_null else 0
            kind = "numérica (texto)" if ok >= 0.9 else "texto"
            if kind == "texto" and _looks_like_date(series):
                kind = "fecha"

        if kind in ("numérica", "numérica (texto)", "binaria"):
            values = to_number(series).dropna()
            if len(values) > 2:
                # asimetria sobre valores recortados al 1%-99% para que un solo atipico no la domine
                low, high = values.quantile([0.01, 0.99])
                skew = float(values.clip(low, high).skew())

        if non_null == 0:
            suggestion, reason = "excluir", "columna vacía"
        elif n_unique <= 1:
            suggestion, reason = "excluir", "valor constante"
        elif missing > cfg["max_missing_col"]:
            suggestion, reason = "excluir", f"{missing:.0%} de faltantes"
        elif kind == "fecha":
            suggestion, reason = "excluir", "fecha (derive variables si las necesita)"
        elif _ID_NAME.search(str(col)):
            suggestion, reason = "excluir", "el nombre parece un identificador"
        elif kind in ("numérica", "numérica (texto)"):
            values = to_number(series).dropna()
            integer_like = bool(np.allclose(values, np.round(values))) if len(values) else False
            consecutive = integer_like and n_unique == non_null and non_null > 20 and values.is_monotonic_increasing
            if consecutive:
                suggestion, reason = "excluir", "enteros consecutivos (parece ID)"
            else:
                suggestion, reason = "numérica", ""
        elif kind == "binaria":
            suggestion, reason = "numérica", "binaria (0/1)"
        else:
            if n_unique == non_null and non_null > 20:
                suggestion, reason = "excluir", "un valor distinto por fila (ID o texto libre)"
            elif n_unique <= cfg["max_categories"]:
                suggestion, reason = "categórica", ""
            else:
                suggestion, reason = "excluir", f"{n_unique} categorías (demasiadas)"

        rows.append({
            "variable": str(col),
            "tipo_detectado": kind,
            "faltantes_%": round(missing * 100, 1),
            "valores_unicos": n_unique,
            "asimetria": round(skew, 2) if not np.isnan(skew) else np.nan,
            "sugerencia": suggestion,
            "motivo": reason,
        })
    return pd.DataFrame(rows)


def resolve_features(df: pd.DataFrame, config: dict | None = None):
    """Devuelve (numericas, categoricas, log) segun CONFIG; 'auto' usa el diagnostico."""
    cfg = merge_config(config)
    diag = diagnose(df, cfg).set_index("variable")
    exclude = {str(c) for c in cfg["exclude"]}

    def pick(value, kind):
        if value == "auto":
            return [c for c in diag.index if diag.loc[c, "sugerencia"] == kind and c not in exclude]
        columns = [str(c) for c in (value or [])]
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(
                f"Estas columnas no existen en el dataset: {missing}.\n"
                f"Columnas disponibles: {list(df.columns)}"
            )
        return [c for c in columns if c not in exclude]

    numeric = pick(cfg["numeric_features"], "numérica")
    categorical = [c for c in pick(cfg["categorical_features"], "categórica") if c not in numeric]

    if cfg["log_features"] == "auto":
        log_features = [
            c for c in numeric
            if diag.loc[c, "asimetria"] > cfg["skew_threshold"] and to_number(df[c]).min() >= 0
        ]
    else:
        log_features = [str(c) for c in (cfg["log_features"] or []) if str(c) in numeric]

    if not numeric and not categorical:
        raise ValueError(
            "No quedó ninguna variable para el modelo. Revise el diagnóstico y escriba "
            "las columnas en CONFIG['numeric_features'] / CONFIG['categorical_features']."
        )
    return numeric, categorical, log_features


def clean_rows(df: pd.DataFrame, numeric, categorical, config: dict | None = None):
    """Limpieza por filas, SOLO para entrenar: duplicados y filas con demasiados faltantes."""
    cfg = merge_config(config)
    data = df.copy()
    report = {"filas_iniciales": int(len(data))}

    if cfg["drop_duplicates"]:
        before = len(data)
        data = data.drop_duplicates()
        report["duplicados_eliminados"] = int(before - len(data))

    parts = []
    if numeric:
        parts.append(data[numeric].apply(to_number))
    if categorical:
        norm = cfg["normalize_categories"]
        parts.append(data[categorical].apply(lambda s: s.map(lambda v: clean_category(v, norm))))
    missing_share = pd.concat(parts, axis=1).isna().mean(axis=1)
    keep = missing_share <= cfg["max_missing_row"]
    report["filas_con_muchos_faltantes"] = int((~keep).sum())

    data = data.loc[keep].reset_index(drop=True)
    report["filas_finales"] = int(len(data))
    if len(data) < 3:
        raise ValueError("Quedan menos de 3 filas después de limpiar. Revise CONFIG['max_missing_row'].")
    return data, report


# ---------------------------------------------------------------------------
# 6. Pipeline: preprocesamiento + K-Means
# ---------------------------------------------------------------------------
def _make_scaler(name):
    return {"standard": StandardScaler(), "robust": RobustScaler(), "minmax": MinMaxScaler(), "none": None}[name]


def _numeric_pipeline(cfg, use_log):
    steps = [
        ("a_numero", NumericCleaner()),
        ("imputar", SimpleImputer(strategy=cfg["numeric_imputer"], keep_empty_features=True)),
    ]
    if use_log:
        steps.append(("log", SignedLog1p()))
    if cfg["outliers"] != "none":
        steps.append(("atipicos", OutlierClipper(
            method=cfg["outliers"], factor=cfg["iqr_factor"], quantiles=tuple(cfg["quantiles"]),
        )))
    scaler = _make_scaler(cfg["scaler"])
    if scaler is not None:
        steps.append(("escalar", scaler))
    return Pipeline(steps)


def build_preprocessor(config, numeric, categorical, log_features) -> ColumnTransformer:
    cfg = merge_config(config)
    plain = [c for c in numeric if c not in log_features]
    transformers = []
    if plain:
        transformers.append(("num", _numeric_pipeline(cfg, use_log=False), plain))
    if log_features:
        transformers.append(("num_log", _numeric_pipeline(cfg, use_log=True), list(log_features)))
    if categorical:
        if cfg["categorical_imputer"] == "constant":
            imputer = SimpleImputer(strategy="constant", fill_value="Desconocido")
        else:
            imputer = SimpleImputer(strategy="most_frequent")
        transformers.append(("cat", Pipeline([
            ("limpiar", CategoricalCleaner(normalize=cfg["normalize_categories"])),
            ("imputar", imputer),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore", max_categories=cfg["max_categories"], sparse_output=False,
            )),
        ]), list(categorical)))
    return ColumnTransformer(transformers, remainder="drop", sparse_threshold=0.0)


def build_model(config, numeric, categorical, log_features, k) -> Pipeline:
    cfg = merge_config(config)
    return Pipeline([
        ("preprocessor", build_preprocessor(cfg, numeric, categorical, log_features)),
        ("cluster", KMeans(n_clusters=int(k), random_state=cfg["random_state"], n_init=cfg["n_init"])),
    ])


# ---------------------------------------------------------------------------
# 7. Eleccion de k y evaluacion
# ---------------------------------------------------------------------------
def _silhouette(X_proc, labels, cfg) -> float:
    if len(np.unique(labels)) < 2:
        return float("nan")
    sample = min(cfg["silhouette_sample"], len(labels))
    return float(silhouette_score(X_proc, labels, sample_size=sample, random_state=cfg["random_state"]))


def evaluate_k(X_proc, config: dict | None = None) -> pd.DataFrame:
    """Prueba k = k_min..k_max y devuelve inercia, silhouette, Davies-Bouldin y Calinski-Harabasz."""
    cfg = merge_config(config)
    n = X_proc.shape[0]
    rows = []
    for k in range(cfg["k_min"], min(cfg["k_max"], n - 1) + 1):
        km = KMeans(n_clusters=k, random_state=cfg["random_state"], n_init=cfg["n_init"]).fit(X_proc)
        labels = km.labels_
        if len(np.unique(labels)) < 2:
            continue
        rows.append({
            "k": k,
            "inercia": float(km.inertia_),
            "silhouette": _silhouette(X_proc, labels, cfg),
            "davies_bouldin": float(davies_bouldin_score(X_proc, labels)),
            "calinski_harabasz": float(calinski_harabasz_score(X_proc, labels)),
            "cluster_min_%": float(np.bincount(labels).min() / n * 100),
        })
    return pd.DataFrame(rows)


def pick_k(k_table: pd.DataFrame, config: dict | None = None) -> int:
    """Mejor silhouette entre los k cuyo cluster mas pequeno no sea diminuto."""
    cfg = merge_config(config)
    if k_table is None or k_table.empty:
        raise ValueError("No se pudo evaluar ningún k. ¿Hay suficientes filas?")
    valid = k_table[k_table["cluster_min_%"] >= cfg["min_cluster_pct"]]
    table = valid if not valid.empty else k_table
    return int(table.loc[table["silhouette"].idxmax(), "k"])


def stability(X_proc, labels, k, config: dict | None = None, repeats=5, fraction=0.8) -> dict:
    """Re-entrena con submuestras del 80% y compara con el modelo final (ARI: 1 = identico)."""
    cfg = merge_config(config)
    rng = np.random.default_rng(cfg["random_state"])
    n = X_proc.shape[0]
    size = max(int(n * fraction), k + 1)
    scores = []
    for i in range(repeats):
        idx = rng.choice(n, size=min(size, n), replace=False)
        km = KMeans(n_clusters=k, random_state=cfg["random_state"] + i + 1, n_init=cfg["n_init"])
        km.fit(X_proc[idx])
        scores.append(adjusted_rand_score(labels, km.predict(X_proc)))
    return {"estabilidad_ari_media": float(np.mean(scores)), "estabilidad_ari_min": float(np.min(scores))}


# ---------------------------------------------------------------------------
# 8. Perfiles e interpretacion automatica
# ---------------------------------------------------------------------------
def cluster_profiles(data, labels, numeric, categorical, normalize=True):
    """Promedios por cluster (unidades originales), z-scores y categoria dominante."""
    labels = np.asarray(labels)
    clusters = sorted(np.unique(labels).tolist())
    counts = pd.Series(labels).value_counts().reindex(clusters)
    profiles = pd.DataFrame(index=pd.Index(clusters, name="cluster"))
    profiles["n_casos"] = counts.values
    profiles["porcentaje"] = (counts.values / len(labels) * 100).round(1)

    z = None
    if numeric:
        num = data[numeric].apply(to_number).reset_index(drop=True)
        means = num.groupby(labels).mean()
        profiles = profiles.join(means.round(3))
        global_std = num.std(ddof=0).replace(0, np.nan)
        z = ((means - num.mean()) / global_std).round(2)

    categorical_info = {}
    for col in categorical:
        cleaned = data[col].map(lambda v: clean_category(v, normalize)).reset_index(drop=True)
        global_share = cleaned.value_counts(normalize=True)
        info = {}
        dominant = []
        for cl in clusters:
            share = cleaned[labels == cl].value_counts(normalize=True)
            if share.empty:
                info[cl] = (None, 0.0, 0.0)
                dominant.append("")
                continue
            mode, mode_share = share.index[0], float(share.iloc[0])
            info[cl] = (mode, mode_share, float(global_share.get(mode, 0.0)))
            dominant.append(f"{mode} ({mode_share:.0%})")
        profiles[f"{col} (más frecuente)"] = dominant
        categorical_info[col] = info
    return profiles, z, categorical_info


def describe_clusters(profiles, z, categorical_info, threshold=0.5, top=3):
    """Genera nombre y descripcion de cada cluster a partir de los datos."""
    names, descriptions = {}, {}
    for cl in profiles.index:
        parts, name_parts = [], []
        if z is not None:
            zc = z.loc[cl].dropna()
            zc = zc.reindex(zc.abs().sort_values(ascending=False).index)
            for feature, value in zc[zc.abs() >= threshold].head(top).items():
                if value > 0:
                    parts.append(f"por encima del promedio en {feature} ({value:+.1f} DE)")
                    name_parts.append(f"↑ {feature}")
                else:
                    parts.append(f"por debajo del promedio en {feature} ({value:+.1f} DE)")
                    name_parts.append(f"↓ {feature}")
        for col, info in categorical_info.items():
            mode, share, global_share = info.get(cl, (None, 0, 0))
            if mode is not None and share - global_share >= 0.15:
                parts.append(f"predomina {col} = {mode} ({share:.0%} vs {global_share:.0%} general)")
                name_parts.append(f"{col}: {mode}")
        n_cases = int(profiles.loc[cl, "n_casos"])
        pct = float(profiles.loc[cl, "porcentaje"])
        body = "; ".join(parts) if parts else "valores cercanos al promedio en todas las variables"
        names[int(cl)] = " · ".join(name_parts[:2]) if name_parts else "Perfil promedio"
        descriptions[int(cl)] = f"{n_cases} casos ({pct:.0f}%): {body[0].upper() + body[1:]}."
    return names, descriptions


# ---------------------------------------------------------------------------
# 9. Entrenamiento completo (lo usan el notebook y train.py)
# ---------------------------------------------------------------------------
def _package_versions() -> dict:
    import sys
    return {
        "python": sys.version.split()[0],
        "scikit-learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    }


def train_clustering(df: pd.DataFrame, config: dict | None = None, verbose: bool = True):
    """Ejecuta TODO el flujo y devuelve (artifact, reporte).

    artifact -> se guarda en model.joblib y lo usa app.py
    reporte  -> datos limpios, etiquetas y matrices para graficar en el notebook
    """
    cfg = merge_config(config)
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    numeric, categorical, log_features = resolve_features(df, cfg)
    data, cleaning = clean_rows(df, numeric, categorical, cfg)
    features = numeric + categorical
    X = data[features]

    # La tabla de k se calcula siempre: sirve para justificar la eleccion en el README
    X_tmp = build_preprocessor(cfg, numeric, categorical, log_features).fit_transform(X)
    k_table = evaluate_k(X_tmp, cfg)
    k = pick_k(k_table, cfg) if cfg["k"] == "auto" else cfg["k"]
    if k >= len(data):
        raise ValueError(f"k = {k} es demasiado grande para {len(data)} filas.")

    model = build_model(cfg, numeric, categorical, log_features, k)
    labels = model.fit_predict(X)
    X_proc = model[:-1].transform(X)

    metrics = {
        "k": int(k),
        "silhouette": _silhouette(X_proc, labels, cfg),
        "davies_bouldin": float(davies_bouldin_score(X_proc, labels)),
        "calinski_harabasz": float(calinski_harabasz_score(X_proc, labels)),
        "inercia": float(model.named_steps["cluster"].inertia_),
    }
    metrics.update(stability(X_proc, labels, k, cfg))

    profiles, z, categorical_info = cluster_profiles(
        data, labels, numeric, categorical, cfg["normalize_categories"])
    auto_names, auto_descriptions = describe_clusters(profiles, z, categorical_info)
    names = {c: cfg["cluster_names"].get(c, auto_names[c]) for c in auto_names}
    descriptions = {c: cfg["cluster_descriptions"].get(c, auto_descriptions[c]) for c in auto_descriptions}

    numeric_values = data[numeric].apply(to_number) if numeric else pd.DataFrame()
    numeric_stats = {}
    for col in numeric:
        values = numeric_values[col].dropna()
        numeric_stats[col] = {
            "min": float(values.min()), "max": float(values.max()),
            "median": float(values.median()), "mean": float(values.mean()),
            "std": float(values.std(ddof=0)),
            "is_integer": bool(np.allclose(values, np.round(values))),
        }
    category_options, category_defaults = {}, {}
    for col in categorical:
        cleaned = data[col].map(lambda v: clean_category(v, cfg["normalize_categories"])).dropna().astype(str)
        category_options[col] = sorted(cleaned.unique().tolist())
        category_defaults[col] = cleaned.value_counts().index[0]

    artifact = {
        # Claves que usa el app.py de la guia
        "model": model,
        "features": features,
        "numeric_features": numeric,
        "categorical_features": categorical,
        "task": "clustering",
        "cluster_descriptions": descriptions,
        # Informacion extra para una app mas completa y para el README
        "title": cfg["title"] or "Segmentación con K-Means",
        "log_features": log_features,
        "n_clusters": int(k),
        "cluster_names": names,
        "category_options": category_options,
        "category_defaults": category_defaults,
        "numeric_stats": numeric_stats,
        "global_means": {c: numeric_stats[c]["mean"] for c in numeric},
        "profiles": profiles.reset_index().to_dict(orient="records"),
        "metrics": metrics,
        "k_table": k_table.to_dict(orient="records") if not k_table.empty else None,
        "cleaning_report": cleaning,
        "config": cfg,
        "n_rows_train": int(len(data)),
        "versions": _package_versions(),
        "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    report = {
        "data": data, "X": X, "X_proc": X_proc, "labels": labels,
        "k_table": k_table, "profiles": profiles, "z": z,
    }
    if verbose:
        print(f"Variables numéricas ({len(numeric)}): {numeric}")
        print(f"Variables categóricas ({len(categorical)}): {categorical}")
        print(f"Con transformación log: {log_features}")
        print(f"Limpieza: {cleaning}")
        print(f"k = {k} | silhouette = {metrics['silhouette']:.3f} | "
              f"estabilidad (ARI medio) = {metrics['estabilidad_ari_media']:.2f}")
    return artifact, report


# ---------------------------------------------------------------------------
# 10. Utilidades para la app
# ---------------------------------------------------------------------------
def example_case(artifact) -> dict:
    """Caso 'tipico': mediana de cada numerica y categoria mas frecuente."""
    case = {}
    for col in artifact["numeric_features"]:
        s = artifact["numeric_stats"][col]
        case[col] = int(round(s["median"])) if s["is_integer"] else round(s["median"], 4)
    for col in artifact["categorical_features"]:
        case[col] = artifact["category_defaults"][col]
    return case


def out_of_range(artifact, values: dict) -> list:
    """Variables numericas cuyo valor esta fuera del rango visto en entrenamiento."""
    outside = []
    for col in artifact["numeric_features"]:
        s = artifact["numeric_stats"][col]
        v = values.get(col)
        if v is not None and (v < s["min"] or v > s["max"]):
            outside.append(col)
    return outside


def assign_clusters(artifact, df: pd.DataFrame) -> pd.DataFrame:
    """Asigna cluster a todas las filas de un DataFrame nuevo (columnas extra se conservan)."""
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    missing = [c for c in artifact["features"] if c not in out.columns]
    if missing:
        raise ValueError(f"Faltan columnas: {missing}")
    labels = artifact["model"].predict(out[artifact["features"]])
    out["cluster"] = labels.astype(int)
    out["cluster_nombre"] = [artifact["cluster_names"].get(int(c), "") for c in labels]
    return out


def save_config(config: dict, path) -> None:
    import json
    cfg = merge_config(config)
    cfg["cluster_names"] = {str(k): v for k, v in cfg["cluster_names"].items()}
    cfg["cluster_descriptions"] = {str(k): v for k, v in cfg["cluster_descriptions"].items()}
    Path(path).write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")


def load_config(path) -> dict:
    import json
    return merge_config(json.loads(Path(path).read_text(encoding="utf-8")))

In [ ]:
import cluster_utils as cu
importlib.reload(cu)  # por si vuelves a ejecutar la celda anterior

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

pd.set_option("display.max_columns", 60)

# Paleta categórica validada para daltonismo (un color fijo por cluster)
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e6e5e0"

def style_axes(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(MUTED)
    ax.tick_params(colors=INK, labelsize=9)
    ax.grid(color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)

## 1. Cargar los datos (guía §5) ✏️
Elige de dónde vienen los datos:

| `DATA_SOURCE` | Qué hace |
|---|---|
| `"subir"` | Abre un selector para subir un CSV o Excel desde tu computador |
| `"url"` | Lee un CSV/Excel público desde `DATA_URL` |
| `"ruta"` | Lee un archivo que ya está en Colab o en Google Drive (`DATA_PATH`) |
| `"demo"` | Dataset *wine* de scikit-learn, solo para practicar el flujo |

El lector detecta solo si el separador es `,` o `;` y si el archivo está en UTF-8 o Latin-1 (típico de Excel en español).

In [ ]:
DATA_SOURCE = "demo"   # ✏️ "subir" | "url" | "ruta" | "demo"
DATA_URL = ""           # si DATA_SOURCE = "url"
DATA_PATH = ""          # si DATA_SOURCE = "ruta", p. ej. "/content/drive/MyDrive/datos.xlsx"
SHEET = 0               # hoja del Excel (nombre o número)

if DATA_SOURCE == "subir":
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))
    df_raw = cu.load_table(filename, sheet_name=SHEET)
elif DATA_SOURCE == "url":
    df_raw = cu.load_table(DATA_URL, sheet_name=SHEET)
elif DATA_SOURCE == "ruta":
    df_raw = cu.load_table(DATA_PATH, sheet_name=SHEET)
elif DATA_SOURCE == "demo":
    from sklearn.datasets import load_wine
    df_raw = load_wine(as_frame=True).frame.drop(columns="target")
else:
    raise ValueError("DATA_SOURCE debe ser 'subir', 'url', 'ruta' o 'demo'.")

# Copia del dataset dentro del proyecto: la usa train.py (guía §12)
df_raw.to_csv(PROJECT / "data" / "datos.csv", index=False)

print(f"{df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
df_raw.head()

> **Datos sensibles:** si el dataset tiene información personal o confidencial, no lo subas a un repositorio público. Borra `data/datos.csv` antes de subir y explica en el README cómo conseguirlo (guía §12).

## 2. Revisión del dataset (guía §6) ✏️
Responde aquí antes de modelar:

- **¿Qué representa una fila?** ✏️
- **¿Cuál es el objetivo del análisis?** ✏️ (p. ej., descubrir perfiles de clientes)
- **¿Qué variables tienen sentido para un caso nuevo?** ✏️ (sin IDs, sin variables posteriores al evento)
- **¿Hay faltantes? ¿Hay numéricas y categóricas?** Mira las tablas de abajo.

In [ ]:
print("Tamaño:", df_raw.shape)
display(df_raw.dtypes.rename("tipo").to_frame().T)
display(df_raw.isna().sum().sort_values(ascending=False).head(15).rename("faltantes").to_frame().T)
display(df_raw.describe(include="all").T)

### Diagnóstico automático de columnas
Para cada columna, el notebook detecta el tipo (incluye números guardados como texto, como `"$ 1.234.567"` o `"45,9"`), el % de faltantes, la asimetría y una **sugerencia**:

- `numérica` / `categórica`: candidata para el modelo.
- `excluir`: identificadores, columnas constantes, fechas, texto libre o columnas con demasiados faltantes.

Es solo una sugerencia: **tú decides** en la sección 3.

In [ ]:
diag = cu.diagnose(df_raw)
diag

## 3. Elegir variables y limpieza ✏️
La limpieza ocurre en **dos niveles**, y conviene entender la diferencia:

| Nivel | Opciones | Qué hace | Cuándo se aplica |
|---|---|---|---|
| **Por filas** | `drop_duplicates`, `max_missing_row` | Elimina filas repetidas o casi vacías | Solo al entrenar |
| **Dentro del pipeline** | `numeric_imputer`, `outliers`, `log_features`, `scaler`, `categorical_imputer`, `normalize_categories` | Convierte texto a número, imputa faltantes, recorta atípicos, aplica log, escala y codifica categorías | Al entrenar **y** en la app con cada caso nuevo |

El segundo nivel va dentro del `Pipeline` para que Streamlit transforme cada caso nuevo **exactamente igual** que en el entrenamiento (guía §7.3).

**Recomendación de la guía (§9):** empieza solo con numéricas interpretables. Las categóricas se pueden agregar (se codifican con one-hot), pero las distancias son más difíciles de interpretar.

In [ ]:
CONFIG = {
    "title": "",                     # ✏️ título de la app, p. ej. "Segmentación de clientes"

    # ---- Variables -------------------------------------------------------
    "numeric_features": "auto",      # ✏️ "auto" (las sugeridas) o una lista: ["edad", "ingreso"]
    "categorical_features": [],      # ✏️ [] = ninguna, "auto" o una lista: ["canal"]
    "log_features": "auto",          # "auto" = numéricas con asimetría > 1 | [] | lista
    "exclude": [],                   # columnas que nunca deben entrar (IDs, variables posteriores...)

    # ---- Limpieza por filas (solo entrenamiento) ------------------------
    "drop_duplicates": True,
    "max_missing_row": 0.5,          # elimina filas con > 50 % de faltantes en las variables elegidas

    # ---- Limpieza dentro del pipeline (también en la app) ---------------
    "numeric_imputer": "median",     # "median" | "mean" | "most_frequent"
    "outliers": "iqr",               # "iqr" | "quantile" | "none"  (recorta, no borra filas)
    "iqr_factor": 1.5,               # límites: Q1 - 1.5·IQR y Q3 + 1.5·IQR
    "quantiles": [0.01, 0.99],       # solo si outliers = "quantile"
    "scaler": "standard",            # "standard" | "robust" | "minmax" | "none"
    "categorical_imputer": "most_frequent",  # "most_frequent" | "constant" ("Desconocido")
    "normalize_categories": True,    # " bogotá " y "BOGOTÁ" -> "Bogotá"
    "max_categories": 15,            # niveles máximos por categórica (el resto se agrupa)

    # ---- Modelo ----------------------------------------------------------
    "k": "auto",                     # se decide en la sección 4
    "k_min": 2,
    "k_max": 8,
    "random_state": 42,
}

### Selector visual (opcional)
Si prefieres no escribir listas, usa este selector: marca varias columnas con **Ctrl/Cmd + clic**, ajusta la limpieza y pulsa **Aplicar**. Actualiza `CONFIG` y te imprime las líneas para copiarlas en la celda anterior, de modo que la decisión quede guardada si vuelves a ejecutar todo.

In [ ]:
import ipywidgets as w

try:
    _num, _cat, _log = cu.resolve_features(df_raw, CONFIG)
except ValueError:
    _num, _cat, _log = [], [], []

_cols = [str(c) for c in df_raw.columns]
_lay = w.Layout(width="32%")
_style = {"description_width": "initial"}

def _multi(label, value):
    return w.SelectMultiple(options=_cols, value=tuple(value), rows=min(12, len(_cols)),
                            description=label, layout=_lay, style=_style)

w_num, w_cat, w_log = _multi("Numéricas", _num), _multi("Categóricas", _cat), _multi("Con log", _log)
w_imp = w.Dropdown(options=["median", "mean", "most_frequent"], value=CONFIG["numeric_imputer"],
                   description="Imputación numérica", layout=_lay, style=_style)
w_out = w.Dropdown(options=["iqr", "quantile", "none"], value=CONFIG["outliers"],
                   description="Atípicos", layout=_lay, style=_style)
w_sc = w.Dropdown(options=["standard", "robust", "minmax", "none"], value=CONFIG["scaler"],
                  description="Escalado", layout=_lay, style=_style)
w_cimp = w.Dropdown(options=["most_frequent", "constant"], value=CONFIG["categorical_imputer"],
                    description="Imputación categórica", layout=_lay, style=_style)
w_norm = w.Checkbox(value=CONFIG["normalize_categories"], description="Normalizar categorías")
w_dup = w.Checkbox(value=CONFIG["drop_duplicates"], description="Eliminar filas duplicadas")
w_miss = w.FloatSlider(value=CONFIG["max_missing_row"], min=0, max=1, step=0.05, readout_format=".0%",
                       description="Máx. faltantes por fila", style=_style, layout=w.Layout(width="50%"))
w_btn = w.Button(description="Aplicar", button_style="success", icon="check")
w_msg = w.Output()

def _apply(_):
    CONFIG.update({
        "numeric_features": list(w_num.value),
        "categorical_features": [c for c in w_cat.value if c not in w_num.value],
        "log_features": [c for c in w_log.value if c in w_num.value],
        "numeric_imputer": w_imp.value, "outliers": w_out.value, "scaler": w_sc.value,
        "categorical_imputer": w_cimp.value, "normalize_categories": w_norm.value,
        "drop_duplicates": w_dup.value, "max_missing_row": round(w_miss.value, 2),
    })
    with w_msg:
        w_msg.clear_output()
        print("✅ CONFIG actualizado. Sigue con la siguiente celda.")
        print("Para dejarlo fijo, copia estas líneas en la celda CONFIG:\n")
        for key in ["numeric_features", "categorical_features", "log_features", "numeric_imputer",
                    "outliers", "scaler", "categorical_imputer", "normalize_categories",
                    "drop_duplicates", "max_missing_row"]:
            print(f'    "{key}": {CONFIG[key]!r},')

w_btn.on_click(_apply)
display(w.VBox([w.HBox([w_num, w_cat, w_log]), w.HBox([w_imp, w_out, w_sc]),
                w.HBox([w_cimp, w_norm, w_dup]), w_miss, w_btn, w_msg]))

### Aplicar la selección
Esta celda muestra exactamente qué entra al modelo y cuántas filas se eliminaron. Si algo no te convence, vuelve a `CONFIG` y ejecuta de nuevo.

In [ ]:
CFG = cu.merge_config(CONFIG)
numeric, categorical, log_features = cu.resolve_features(df_raw, CFG)
df_clean, cleaning = cu.clean_rows(df_raw, numeric, categorical, CFG)

print("Numéricas    :", numeric)
print("Categóricas  :", categorical)
print("Con log      :", log_features)
print("Limpieza por filas:", cleaning)

X = df_clean[numeric + categorical]
preprocessor = cu.build_preprocessor(CFG, numeric, categorical, log_features)
X_proc = preprocessor.fit_transform(X)
print(f"\nMatriz que verá K-Means: {X_proc.shape[0]} filas × {X_proc.shape[1]} columnas")
X.head()

## 4. Elegir el número de clusters k (guía §9.3)
Se prueba cada k entre `k_min` y `k_max`:

| Métrica | Cómo leerla |
|---|---|
| **Inercia** | Siempre baja al subir k; busca el *codo*, donde deja de bajar mucho |
| **Silhouette** | De -1 a 1; **más alto** = clusters más separados |
| **Davies-Bouldin** | **Más bajo** es mejor |
| **cluster_min_%** | Tamaño del cluster más pequeño; uno diminuto suele ser un grupo de atípicos |

El k sugerido es el de mayor silhouette sin clusters diminutos. **No es una verdad absoluta** (guía §9.3): si un k vecino da perfiles más interpretables, puedes elegirlo y justificarlo.

In [ ]:
k_table = cu.evaluate_k(X_proc, CFG)
k_sugerido = cu.pick_k(k_table, CFG)
display(k_table.round(3).style.highlight_max(subset=["silhouette"], color="#cde2fb")
        .highlight_min(subset=["davies_bouldin"], color="#cde2fb"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in [(axes[0], "inercia", "Método del codo: inercia"),
                       (axes[1], "silhouette", "Silhouette (más alto = mejor)")]:
    ax.plot(k_table["k"], k_table[col], marker="o", markersize=7, color=PALETTE[0], linewidth=2)
    ax.axvline(k_sugerido, color=MUTED, linestyle="--", linewidth=1)
    ax.annotate(f"k sugerido = {k_sugerido}", xy=(k_sugerido, ax.get_ylim()[0]), xytext=(5, 6),
                textcoords="offset points", color=INK, fontsize=9)
    ax.set_title(title, loc="left", color=INK)
    ax.set_xlabel("k (número de clusters)")
    ax.set_xticks(k_table["k"])
    style_axes(ax)
plt.tight_layout()
plt.show()
print("k sugerido:", k_sugerido)

In [ ]:
CONFIG["k"] = k_sugerido   # ✏️ o escribe el número que elijas, p. ej. CONFIG["k"] = 4
print("k elegido:", CONFIG["k"])

## 5. Entrenar el pipeline final y evaluar
`cu.train_clustering` ejecuta **todo** el flujo (variables → limpieza → pipeline → K-Means → métricas → perfiles). Es la misma función que usará `train.py`, así que notebook y script producen el mismo modelo.

Además de silhouette y Davies-Bouldin se calcula la **estabilidad**: se reentrena con 5 submuestras del 80 % y se compara con el modelo final mediante el ARI (1 = mismos clusters; menos de ~0.7 indica clusters frágiles).

In [ ]:
artifact, rep = cu.train_clustering(df_raw, CONFIG)
display(artifact["model"])   # diagrama del pipeline completo
pd.Series(artifact["metrics"]).round(3).to_frame("valor")

In [ ]:
from sklearn.decomposition import PCA

labels, X_proc = rep["labels"], rep["X_proc"]
k = artifact["n_clusters"]
names = artifact["cluster_names"]
markers = ["o", "s", "^", "D"]

if X_proc.shape[1] >= 2:
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X_proc)
    centers = pca.transform(artifact["model"].named_steps["cluster"].cluster_centers_)
    xlab = f"Componente 1 ({pca.explained_variance_ratio_[0]:.0%} de la varianza)"
    ylab = f"Componente 2 ({pca.explained_variance_ratio_[1]:.0%} de la varianza)"
else:
    coords = np.column_stack([X_proc[:, 0], np.zeros(len(X_proc))])
    centers = np.column_stack([artifact["model"].named_steps["cluster"].cluster_centers_[:, 0], np.zeros(k)])
    xlab, ylab = "Variable escalada", ""

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [3, 2]})
for c in range(k):
    m = labels == c
    ax.scatter(coords[m, 0], coords[m, 1], s=16, alpha=0.65, color=PALETTE[c % 8],
               marker=markers[(c // 8) % 4], edgecolors="white", linewidths=0.4,
               label=f"{c} · {names[c]}")
ax.scatter(centers[:, 0], centers[:, 1], s=140, marker="X", color=INK, edgecolors="white",
           linewidths=1.5, label="Centro del cluster")
ax.set_title("Clusters proyectados en 2D (PCA)", loc="left", color=INK)
ax.set_xlabel(xlab); ax.set_ylabel(ylab)
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
style_axes(ax)

sizes = pd.Series(labels).value_counts().sort_index()
bars = ax2.barh([f"Cluster {c}" for c in sizes.index], sizes.values,
                color=[PALETTE[c % 8] for c in sizes.index], height=0.6)
for bar, v in zip(bars, sizes.values):
    ax2.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
             f"  {v} ({v / sizes.sum():.0%})", va="center", fontsize=9, color=INK)
ax2.invert_yaxis()
ax2.set_title("Tamaño de cada cluster", loc="left", color=INK)
ax2.set_xlabel("Número de casos")
style_axes(ax2)
ax2.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## 6. Interpretar los clusters (guía §10) ✏️
Un cluster llamado 0, 1 o 2 no significa nada por sí mismo. Abajo tienes:

1. **Mapa de calor**: cuántas desviaciones estándar (DE) está cada cluster por encima (rojo) o por debajo (azul) del promedio general en cada variable.
2. **Tabla de perfiles**: promedios en unidades originales y categoría más frecuente.
3. **Nombres y descripciones automáticas**, generados a partir de los datos.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

z = rep["z"]
if z is not None and not z.empty:
    cmap = LinearSegmentedColormap.from_list("divergente", ["#2a78d6", "#f0efec", "#e34948"])
    lim = max(1.0, float(np.nanmax(np.abs(z.values))))
    fig, ax = plt.subplots(figsize=(1.2 * z.shape[1] + 3, 0.6 * z.shape[0] + 2))
    im = ax.imshow(z.values, cmap=cmap, vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(z.shape[1]), z.columns, rotation=30, ha="right")
    ax.set_yticks(range(z.shape[0]), [f"Cluster {c}" for c in z.index])
    for i in range(z.shape[0]):
        for j in range(z.shape[1]):
            ax.text(j, i, f"{z.values[i, j]:+.1f}", ha="center", va="center", fontsize=9, color=INK)
    fig.colorbar(im, ax=ax, shrink=0.8, label="DE respecto al promedio general")
    ax.set_title("Perfil de cada cluster (variables numéricas)", loc="left", color=INK)
    for s in ax.spines.values():
        s.set_visible(False)
    plt.tight_layout()
    plt.show()

display(rep["profiles"])

In [ ]:
print("Nombres y descripciones generados automáticamente:\n")
for c in range(artifact["n_clusters"]):
    print(f"Cluster {c}: {artifact['cluster_names'][c]}")
    print(f"   {artifact['cluster_descriptions'][c]}\n")

print("-" * 70)
print("Plantilla para copiar en la celda siguiente y reescribir con tus palabras:\n")
print("CLUSTER_NAMES = {")
for c in range(artifact["n_clusters"]):
    print(f'    {c}: "{artifact["cluster_names"][c]}",')
print("}\nCLUSTER_DESCRIPTIONS = {")
for c in range(artifact["n_clusters"]):
    print(f'    {c}: "{artifact["cluster_descriptions"][c]}",')
print("}")

✏️ **Pon nombres que surjan de los datos** (guía §10), por ejemplo «Jóvenes de alta frecuencia y ticket bajo». Si dejas los diccionarios vacíos se usan los automáticos.

Los números de cluster dependen de los datos y de `random_state`. Si cambias variables, limpieza o k, vuelve a revisar los nombres.

In [ ]:
CLUSTER_NAMES = {}          # ✏️ p. ej. {0: "Compradores frecuentes", 1: "...", 2: "..."}
CLUSTER_DESCRIPTIONS = {}   # ✏️ p. ej. {0: "Clientes jóvenes que compran seguido por la app...", ...}

CONFIG["cluster_names"] = CLUSTER_NAMES
CONFIG["cluster_descriptions"] = CLUSTER_DESCRIPTIONS

## 7. Guardar `model.joblib` y `config.json` (guía §8.2 y §16)
- `model.joblib` contiene el pipeline y la información que necesita la app: variables, rangos, categorías permitidas, perfiles y descripciones.
- `config.json` guarda **tus decisiones** para que `train.py` reconstruya exactamente el mismo modelo.

In [ ]:
artifact, rep = cu.train_clustering(df_raw, CONFIG, verbose=False)
joblib.dump(artifact, PROJECT / "model.joblib")
cu.save_config({**CONFIG, "k": artifact["n_clusters"]}, PROJECT / "config.json")

print("✅ Guardados model.joblib y config.json")
for c in range(artifact["n_clusters"]):
    print(f"  Cluster {c}: {artifact['cluster_names'][c]}")

### Pruebas mínimas antes de Docker (guía §16)
Se recarga el modelo **desde el archivo** (sin reentrenar) y se prueban varios casos: filas reales, un caso típico, un caso extremo y un caso con faltantes.

In [ ]:
loaded = joblib.load(PROJECT / "model.joblib")
feats = loaded["features"]

typical = cu.example_case(loaded)
extreme = {f: loaded["numeric_stats"][f]["max"] * 10 for f in loaded["numeric_features"]}
extreme.update({f: loaded["category_options"][f][0] for f in loaded["categorical_features"]})
empty = {f: None for f in feats}

real_rows = rep["data"][feats].sample(min(3, len(rep["data"])), random_state=1).to_dict("records")
tests = pd.DataFrame(real_rows + [typical, extreme, empty], columns=feats)
tests.insert(0, "caso", ["fila real"] * len(real_rows)
             + ["caso típico (medianas)", "valores extremos (×10 el máximo)", "todo faltante"])

tests["cluster"] = loaded["model"].predict(tests[feats])
tests["nombre"] = tests["cluster"].map(loaded["cluster_names"])
print("Columnas que espera el modelo:", feats)
tests[["caso"] + feats + ["cluster", "nombre"]]

## 8. `train.py`: el entrenamiento como script reproducible (guía §11)
Lee `data/datos.csv` y `config.json` y reconstruye `model.joblib`. Cualquier persona puede regenerar el modelo con:
```bash
python train.py
```

In [ ]:
%%writefile {PROJECT}/train.py
"""
train.py - reconstruye el modelo de clustering de forma reproducible.

Uso:
    python train.py
    python train.py --data data/otro.csv --config config.json --output model.joblib

Lee el dataset y las decisiones guardadas en config.json (variables, limpieza, k,
nombres de clusters), entrena el pipeline y guarda model.joblib.
"""
import argparse
import sys
from pathlib import Path

import joblib

BASE_DIR = Path(__file__).resolve().parent
sys.path.insert(0, str(BASE_DIR))

import cluster_utils as cu  # noqa: E402

DATA_PATH = BASE_DIR / "data" / "datos.csv"
CONFIG_PATH = BASE_DIR / "config.json"
MODEL_PATH = BASE_DIR / "model.joblib"


def load_data(path: Path):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"No se encontró el dataset en {path}. Revise el README para saber cómo obtenerlo."
        )
    return cu.load_table(path)


def train_model(df, config: dict):
    # 1. Variables  2. Limpieza  3. Preprocesamiento  4. Entrenar  5. Evaluar  6. Artifact
    artifact, _ = cu.train_clustering(df, config, verbose=True)
    return artifact


def main() -> None:
    parser = argparse.ArgumentParser(description="Entrena el modelo de clustering (Ruta B).")
    parser.add_argument("--data", default=DATA_PATH, type=Path)
    parser.add_argument("--config", default=CONFIG_PATH, type=Path)
    parser.add_argument("--output", default=MODEL_PATH, type=Path)
    args = parser.parse_args()

    config = cu.load_config(args.config) if args.config.exists() else {}
    if not config:
        print("Aviso: no se encontró config.json; se usará la configuración automática.")

    df = load_data(args.data)
    artifact = train_model(df, config)
    joblib.dump(artifact, args.output)

    m = artifact["metrics"]
    print(f"\nModelo guardado en {args.output}")
    print(f"k = {m['k']} | silhouette = {m['silhouette']:.3f} | "
          f"Davies-Bouldin = {m['davies_bouldin']:.3f}")
    for c, name in artifact["cluster_names"].items():
        print(f"  Cluster {c}: {name}")


if __name__ == "__main__":
    main()

In [ ]:
import subprocess
subprocess.run([sys.executable, "train.py"], cwd=PROJECT, check=True)

In [ ]:
# Comprobación: el modelo de train.py asigna los mismos clusters que el del notebook
from_script = joblib.load(PROJECT / "model.joblib")
match = (from_script["model"].predict(rep["X"]) == rep["labels"]).mean()
print(f"Coincidencia notebook vs train.py: {match:.1%}")
assert match == 1.0, "train.py no reproduce el modelo: revisa config.json y data/datos.csv"

## 9. `app.py`: la interfaz Streamlit (guía §13 y §14)
La app construye el formulario **a partir del artifact**, así que sirve para cualquier dataset sin cambiar código:
- **Numéricas**: `number_input` que empieza en la mediana y muestra el rango del entrenamiento.
- **Categóricas**: `selectbox` solo con los valores vistos al entrenar (guía §14).
- **Salida**: cluster asignado, nombre, descripción, distancia a cada centro y comparación con el perfil.
- **Pestaña "Varios casos"**: sube un CSV/Excel, asigna cluster a todas las filas y permite descargar el resultado.

In [ ]:
%%writefile {PROJECT}/app.py
"""
app.py - interfaz Streamlit de la Ruta B (clustering con K-Means).

Carga model.joblib (NO vuelve a entrenar), recibe un caso nuevo o un archivo
con varios casos y devuelve el cluster asignado con su descripcion.
"""
import sys
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

BASE_DIR = Path(__file__).resolve().parent
sys.path.insert(0, str(BASE_DIR))

import cluster_utils as cu  # noqa: E402  (necesario para cargar las clases del pipeline)

st.set_page_config(page_title="Segmentación K-Means", page_icon="🧩", layout="wide")


@st.cache_resource
def load_artifact():
    return joblib.load(BASE_DIR / "model.joblib")


artifact = load_artifact()

model = artifact["model"]
features = artifact["features"]
numeric_features = artifact.get("numeric_features", [])
categorical_features = artifact.get("categorical_features", [])
task = artifact["task"]
names = artifact.get("cluster_names", {})
descriptions = artifact.get("cluster_descriptions", {})
stats = artifact.get("numeric_stats", {})
options = artifact.get("category_options", {})
defaults = artifact.get("category_defaults", {})
metrics = artifact.get("metrics", {})
profiles = pd.DataFrame(artifact.get("profiles", []))
if not profiles.empty:
    profiles = profiles.set_index("cluster")


def cluster_label(c: int) -> str:
    name = names.get(int(c), "")
    return f"Cluster {c} · {name}" if name else f"Cluster {c}"


def fmt(x) -> str:
    return f"{x:,.4g}"


# ---------------------------------------------------------------------------
st.title(artifact.get("title", "Segmentación con K-Means"))
st.write(
    "Ingrese las características de un caso nuevo y el modelo lo asignará al grupo "
    "(cluster) más parecido. El modelo no se reentrena: aplica el mismo pipeline "
    "de limpieza y escalado que se usó en el entrenamiento."
)

with st.sidebar:
    st.header("Modelo")
    st.metric("Número de clusters (k)", artifact.get("n_clusters", "-"))
    if "silhouette" in metrics:
        st.metric("Silhouette", f"{metrics['silhouette']:.3f}",
                  help="De -1 a 1. Más alto = clusters más separados.")
    if "estabilidad_ari_media" in metrics:
        st.metric("Estabilidad (ARI)", f"{metrics['estabilidad_ari_media']:.2f}",
                  help="Qué tanto se repiten los clusters al reentrenar con submuestras. 1 = idénticos.")
    st.metric("Filas de entrenamiento", artifact.get("n_rows_train", "-"))
    st.caption("Variables: " + ", ".join(features))

tab_case, tab_batch, tab_profiles = st.tabs(["Un caso", "Varios casos (archivo)", "Perfiles de los clusters"])

# ---------------------------------------------------------------------------
with tab_case:
    with st.form("nuevo_caso"):
        values = {}
        columns = st.columns(2)
        for i, feature in enumerate(numeric_features):
            s = stats.get(feature)
            col = columns[i % 2]
            if s is None:
                values[feature] = col.number_input(feature, value=0.0)
                continue
            help_text = f"En entrenamiento: mín {fmt(s['min'])} · mediana {fmt(s['median'])} · máx {fmt(s['max'])}"
            if s.get("is_integer"):
                values[feature] = col.number_input(feature, value=int(round(s["median"])), step=1, help=help_text)
            else:
                values[feature] = col.number_input(feature, value=float(s["median"]), help=help_text)
        for j, feature in enumerate(categorical_features):
            opts = options.get(feature, [])
            col = columns[(len(numeric_features) + j) % 2]
            if opts:
                default = defaults.get(feature)
                index = opts.index(default) if default in opts else 0
                values[feature] = col.selectbox(feature, opts, index=index)
            else:
                values[feature] = col.text_input(feature)
        submitted = st.form_submit_button("Ejecutar modelo", type="primary")

    if submitted:
        X_new = pd.DataFrame([values], columns=features)
        result = int(model.predict(X_new)[0])

        if task == "clustering":
            st.success(f"Cluster asignado: **{cluster_label(result)}**")
            st.info(descriptions.get(result, "Sin descripción"))

        outside = cu.out_of_range(artifact, values)
        if outside:
            st.warning(
                "Valores fuera del rango visto en entrenamiento: " + ", ".join(outside)
                + ". El pipeline los recorta, pero la asignación puede ser menos confiable."
            )

        left, right = st.columns(2)
        distances = model.transform(X_new)[0]
        dist_df = pd.DataFrame(
            {"Distancia al centro": distances},
            index=[cluster_label(c) for c in range(len(distances))],
        )
        left.write("**Distancia al centro de cada cluster** (menor = más parecido)")
        left.bar_chart(dist_df, horizontal=True)

        if numeric_features and not profiles.empty:
            comparison = pd.DataFrame({
                "Este caso": [values[f] for f in numeric_features],
                f"Promedio cluster {result}": [profiles.loc[result, f] for f in numeric_features],
                "Promedio general": [artifact["global_means"][f] for f in numeric_features],
            }, index=numeric_features)
            right.write("**Comparación con el perfil del cluster**")
            right.dataframe(comparison.round(2))

# ---------------------------------------------------------------------------
with tab_batch:
    st.write(
        "Suba un CSV o Excel con estas columnas: **" + ", ".join(features) + "**. "
        "Las columnas adicionales se conservan en el resultado."
    )
    uploaded = st.file_uploader("Archivo", type=["csv", "xlsx", "xls"])
    if uploaded is not None:
        try:
            df_in = cu.load_table(uploaded, name=uploaded.name)
            result_df = cu.assign_clusters(artifact, df_in)
        except Exception as exc:  # mensaje claro en lugar de un traceback
            st.error(f"No se pudo procesar el archivo: {exc}")
        else:
            counts = result_df["cluster"].map(cluster_label).value_counts().sort_index()
            st.write(f"Se asignaron **{len(result_df)}** casos.")
            st.bar_chart(counts.rename("Casos"), horizontal=True)
            st.dataframe(result_df.head(500))
            st.download_button(
                "Descargar resultados (CSV)",
                result_df.to_csv(index=False).encode("utf-8"),
                file_name="resultados_clusters.csv",
                mime="text/csv",
            )

# ---------------------------------------------------------------------------
with tab_profiles:
    st.write("Promedios de cada cluster en las unidades originales de los datos.")
    if not profiles.empty:
        st.dataframe(profiles)
    for c in sorted(descriptions):
        st.markdown(f"**{cluster_label(c)}** — {descriptions[c]}")

## 10. Probar Streamlit en Colab (guía §15)
Se levanta la app en segundo plano y se revisa el log.

In [ ]:
import subprocess, time

# detiene una ejecución anterior
if "streamlit_proc" in globals() and streamlit_proc.poll() is None:
    streamlit_proc.terminate()
    streamlit_proc.wait()
streamlit_log = open(BASE / "streamlit.log", "w")
streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(PROJECT / "app.py"),
     "--server.port", "8501", "--server.address", "0.0.0.0", "--server.headless", "true"],
    stdout=streamlit_log, stderr=subprocess.STDOUT,
)
# Espera hasta 30 s a que Streamlit termine de arrancar
for _ in range(30):
    time.sleep(1)
    log_text = (BASE / "streamlit.log").read_text(encoding="utf-8", errors="replace")
    if "You can now view" in log_text or streamlit_proc.poll() is not None:
        break
print(log_text)
print("✅ Streamlit está corriendo" if "You can now view" in log_text and streamlit_proc.poll() is None
      else "❌ Streamlit no arrancó: revisa el log de arriba")

Para verla en el navegador se usa un túnel temporal (**no es el despliegue final**).

1. Ejecuta la celda: primero muestra una **contraseña** (es la IP pública de Colab) y después una URL `https://….loca.lt`.
2. Abre la URL y pega la contraseña en la página de aviso de localtunnel.
3. Prueba al menos **tres casos distintos** (guía §16). Para cerrar el túnel, detén la celda (■).

Si la red bloquea el túnel, no pasa nada: si el log dice `You can now view your Streamlit app`, continúa con Docker y GCP.

In [ ]:
if IN_COLAB:
    # Túnel temporal solo necesario en Colab
    get_ipython().system("npm install -g localtunnel > /dev/null 2>&1")
    print("Contraseña del túnel:")
    get_ipython().system("curl -s https://loca.lt/mytunnelpassword")
    print()
    get_ipython().system("npx localtunnel --port 8501")
else:
    print("En local no hace falta túnel: abre http://localhost:8501")


## 11. Archivos para Docker y README (guía §17 a §20)
`requirements.txt` se genera con las **versiones exactas instaladas en Colab** y el `Dockerfile` usa la misma versión de Python. Así se evita el error clásico de un `model.joblib` creado con una versión de scikit-learn que no carga con otra.

In [ ]:
from importlib.metadata import version

packages = ["streamlit", "pandas", "numpy", "scikit-learn", "joblib", "openpyxl"]
(PROJECT / "requirements.txt").write_text(
    "\n".join(f"{p}=={version(p)}" for p in packages) + "\n", encoding="utf-8")

(PROJECT / ".dockerignore").write_text("""\
.git
.venv
__pycache__
*.pyc
.ipynb_checkpoints
.env
*.log
*.zip
""", encoding="utf-8")

PY = f"{sys.version_info.major}.{sys.version_info.minor}"
(PROJECT / "Dockerfile").write_text(f"""\
FROM python:{PY}-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

CMD ["streamlit", "run", "app.py", "--server.address=0.0.0.0", "--server.port=8501", "--server.headless=true"]
""", encoding="utf-8")

for name in ["requirements.txt", ".dockerignore", "Dockerfile"]:
    print(f"===== {name} =====")
    print((PROJECT / name).read_text())

### README.md (guía §20)
Se genera con los datos reales del modelo: variables, limpieza, métricas, perfiles y un ejemplo de entrada y salida. **Completa las partes marcadas con ✏️** (autores, objetivo, descripción del dataset y justificación) antes de entregar.

In [ ]:
import json

def md_table(df, digits=3):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for row in df.itertuples(index=False, name=None):
        cells = [f"{v:,.{digits}f}" if isinstance(v, (float, np.floating)) else str(v) for v in row]
        lines.append("| " + " | ".join(c.replace("|", "/") for c in cells) + " |")
    return "\n".join(lines)

art = joblib.load(PROJECT / "model.joblib")
cfg, met = art["config"], art["metrics"]
diag_final = cu.diagnose(df_raw, cfg)
excluded = diag_final[~diag_final["variable"].isin(art["features"])][["variable", "motivo"]]
excluded["motivo"] = excluded["motivo"].replace("", "no seleccionada")
example_in = cu.example_case(art)
example_out = int(art["model"].predict(pd.DataFrame([example_in])[art["features"]])[0])
profiles_md = md_table(pd.DataFrame(art["profiles"]), digits=2)
clusters_md = md_table(pd.DataFrame({
    "Cluster": list(art["cluster_names"]),
    "Nombre": list(art["cluster_names"].values()),
    "Descripción": [art["cluster_descriptions"][c] for c in art["cluster_names"]],
}))
k_md = md_table(pd.DataFrame(art["k_table"])) if art["k_table"] else "(k fijado manualmente)"
var_lines = "\n".join(
    f"- `{c}` (numérica{', con log' if c in art['log_features'] else ''})" for c in art["numeric_features"]
) + "\n" + "\n".join(f"- `{c}` (categórica, one-hot)" for c in art["categorical_features"])

readme = f"""# {art['title']}

**Autores:** ✏️ Nombre 1 · Nombre 2

## Objetivo
✏️ Describe en 2 o 3 líneas qué perfiles se quieren descubrir y para qué sirven.

## Dataset
- **Descripción y fuente:** ✏️
- **Unidad de observación (qué es una fila):** ✏️
- **Tamaño:** {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas ({art['n_rows_train']} filas tras la limpieza).
- **Ubicación:** `data/datos.csv`. ✏️ Si el dataset no se puede publicar, explica aquí cómo obtenerlo y dónde ubicarlo.

## Ruta elegida
**Ruta B: aprendizaje no supervisado.** Segmentación con K-Means. No se usa variable objetivo.

## Variables utilizadas
{var_lines}

Variables excluidas:

{md_table(excluded) if len(excluded) else "Ninguna."}

## Preparación de datos
Limpieza por filas (solo entrenamiento):
- Eliminar filas duplicadas: {'sí' if cfg['drop_duplicates'] else 'no'} (filas eliminadas: {art['cleaning_report'].get('duplicados_eliminados', 0)}).
- Eliminar filas con más del {cfg['max_missing_row']:.0%} de faltantes (filas eliminadas: {art['cleaning_report']['filas_con_muchos_faltantes']}).

Dentro del pipeline (se aplica igual en entrenamiento y en la app):
1. Conversión de texto a número (`"$ 1.234.567"`, `"45,9"`, `"15%"`).
2. Imputación de faltantes numéricos: `{cfg['numeric_imputer']}`.
3. Transformación log en variables sesgadas: {art['log_features'] or 'ninguna'}.
4. Tratamiento de atípicos (recorte): `{cfg['outliers']}`.
5. Escalado: `{cfg['scaler']}`.
6. Categóricas: limpieza de texto, imputación `{cfg['categorical_imputer']}` y one-hot encoding.

## Modelo y justificación
K-Means con **k = {art['n_clusters']}** (`random_state = {cfg['random_state']}`, `n_init = {cfg['n_init']}`). K-Means permite asignar casos nuevos con `predict()`, que es lo que necesita la app.

✏️ Justifica la elección de k con la tabla siguiente (codo, silhouette y tamaño mínimo de cluster) y con la interpretabilidad de los perfiles.

{k_md}

## Evaluación
| Métrica | Valor |
|---|---|
| Silhouette | {met['silhouette']:.3f} |
| Davies-Bouldin | {met['davies_bouldin']:.3f} |
| Calinski-Harabasz | {met['calinski_harabasz']:.1f} |
| Inercia | {met['inercia']:.1f} |
| Estabilidad (ARI medio en submuestras del 80 %) | {met['estabilidad_ari_media']:.2f} |

## Interpretación de la salida
La app devuelve el **cluster asignado** y la descripción de su perfil:

{clusters_md}

Perfiles promedio (unidades originales):

{profiles_md}

## Cómo ejecutar
### Entrenar
```bash
pip install -r requirements.txt
python train.py
```
### Streamlit (local)
```bash
streamlit run app.py
```
### Docker
```bash
docker build -t ml-streamlit-app .
docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
```
Abrir `http://localhost:8501` (o `http://IP_EXTERNA_DE_LA_VM:8501` en GCP, con el puerto 8501 abierto en el firewall).

### Google Cloud (VM Ubuntu 24.04)
```bash
sudo apt update && sudo apt install -y docker.io git
sudo systemctl enable --now docker
git clone <URL_DEL_REPOSITORIO>
cd <NOMBRE_DEL_REPOSITORIO>
sudo docker build -t ml-streamlit-app .
sudo docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
```

## Ejemplo de entrada y salida
Entrada (caso típico: medianas y categorías más frecuentes):
```json
{json.dumps(example_in, ensure_ascii=False, indent=2)}
```
Salida esperada: **Cluster {example_out}: {art['cluster_names'][example_out]}**

## Estructura
```
app.py            interfaz Streamlit
train.py          entrenamiento reproducible
cluster_utils.py  limpieza, pipeline y perfiles
config.json       decisiones de variables, limpieza y k
model.joblib      pipeline entrenado
requirements.txt  dependencias con versión fija
Dockerfile, .dockerignore
data/datos.csv
```

Versiones usadas: {', '.join(f'{k} {v}' for k, v in art['versions'].items())}.
"""

(PROJECT / "README.md").write_text(readme, encoding="utf-8")
print("✅ README.md generado. Vista previa:\n")
print(readme[:1500], "...")

## 12. Revisión final y descarga (guía §19 y §21)

In [ ]:
import zipfile, shutil

for cache in PROJECT.rglob("__pycache__"):
    shutil.rmtree(cache, ignore_errors=True)

required = ["app.py", "train.py", "cluster_utils.py", "config.json", "model.joblib",
            "requirements.txt", "Dockerfile", ".dockerignore", "README.md", "data/datos.csv"]
for name in required:
    path = PROJECT / name
    print(("✅" if path.exists() else "❌"), name, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "")

zip_path = BASE / "ml_streamlit_project.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(PROJECT.rglob("*")):
        if path.is_file() and ".ipynb_checkpoints" not in path.parts and path.suffix != ".log":
            zf.write(path, Path(PROJECT.name) / path.relative_to(PROJECT))
print(f"\n📦 {zip_path} ({zip_path.stat().st_size / 1024:.0f} KB)")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))

**Subir a GitHub (guía §21):** crea el repositorio desde la web de GitHub, descomprime el .zip y arrastra **todos** los archivos, incluido `.dockerignore`. Los archivos que empiezan por punto quedan ocultos en macOS y Windows: actívalos para verlos o súbelo aparte. No subas tokens, `.env` ni datos sensibles.

> Si `model.joblib` pesa más de 25 MB, la web de GitHub no deja subirlo. En ese caso usa Git desde la terminal (permite hasta 100 MB) o añade `RUN python train.py` al Dockerfile para que la imagen entrene el modelo al construirse.

## 13. Despliegue en Google Cloud (guía §22 a §30)
Resumen de comandos para la terminal **SSH de la VM** (Ubuntu 24.04, con IP externa y la etiqueta de red `streamlit-server`):

```bash
# 1. Docker y Git
sudo apt update
sudo apt install -y docker.io git
sudo systemctl enable --now docker

# 2. Clonar el repositorio
git clone <URL_DEL_REPOSITORIO>
cd <NOMBRE_DEL_REPOSITORIO>
ls -la            # deben aparecer Dockerfile, app.py, cluster_utils.py, model.joblib...

# 3. Construir la imagen y ejecutar el contenedor
sudo docker build -t ml-streamlit-app .
sudo docker run --rm -p 8501:8501 ml-streamlit-app        # prueba; Ctrl+C para detener
sudo docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
sudo docker ps
sudo docker logs ml-streamlit
```

Regla de firewall (desde **Cloud Shell**, o en la consola: *VPC network → Firewall*):
```bash
gcloud compute firewall-rules create allow-streamlit-8501 \
  --allow=tcp:8501 --direction=INGRESS --target-tags=streamlit-server
```
Abrir `http://IP_EXTERNA_DE_LA_VM:8501`.

**Actualizar después de cambios** (guía §30):
```bash
git pull
sudo docker stop ml-streamlit && sudo docker rm ml-streamlit
sudo docker build -t ml-streamlit-app .
sudo docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
```

💸 **Al terminar, detén o elimina la VM** para que no siga generando costos.

### Errores frecuentes
| Síntoma | Revisión |
|---|---|
| `ModuleNotFoundError: cluster_utils` al cargar el modelo | `cluster_utils.py` debe estar en el repositorio, junto a `app.py` |
| Error o advertencia de versión al cargar `model.joblib` | Usa el `requirements.txt` generado (versiones fijas), no uno escrito a mano |
| Funciona en la VM pero no desde fuera | Regla de firewall, etiqueta de red `streamlit-server` en la VM e IP externa |
| El contenedor se detiene | `sudo docker ps -a` y `sudo docker logs ml-streamlit` |